# HHI / Shannon / Top-1 Correlation Analysis

**Purpose:**  
This notebook **empirically validates** that claim on our data and extends  
it to Shannon. The goals:

1. Compute and report the full correlation matrix of all concentration metrics.
2. Show that HHI-Top1 correlation is indeed ~0.9 (defending the decision to  
   exclude Top-1 from the index).
3. Assess whether Shannon adds **independent information** beyond HHI — if the  
   Shannon-HHI correlation is meaningfully lower than Top1-HHI, Shannon is  
   worth keeping.
4. Produce the correlation matrix figure for the methods section.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="white", font_scale=1.1)

# ---- Paths ----
ROOT = Path(".").resolve().parent
TRADE_MATRIX = ROOT / "data" / "cleaned" / "trade_matrix_cleaned.csv"
CONC_PATH = ROOT / "data" / "cleaned" / "concentration_with_shannon.csv"
VIZ_DIR = ROOT / "visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# ---- Item-to-commodity mapping ----
ITEM_MAP = {
    "Wheat": "Wheat",
    "Wheat and meslin flour": "Wheat",
    "Rice, paddy (rice milled equivalent)": "Rice",
    "Rice, milled": "Rice",
    "Maize (corn)": "Maize",
}

COMMODITIES = ["Wheat", "Rice", "Maize"]

print(f"Concentration CSV available: {CONC_PATH.exists()}")
print(f"Trade matrix available: {TRADE_MATRIX.exists()}")

## 1. Load or Compute Concentration Metrics

In [ ]:
if CONC_PATH.exists():
    print(f"Loading from {CONC_PATH.name}")
    conc = pd.read_csv(CONC_PATH)
else:
    print("Computing from trade_matrix_cleaned.csv...")
    raw = pd.read_csv(TRADE_MATRIX)
    imp = raw[
        (raw["Element"] == "Import quantity")
        & (raw["Unit"] == "t")
        & (raw["Value"] > 0)
    ].copy()
    imp["Commodity"] = imp["Item"].map(ITEM_MAP)
    imp = imp[imp["Commodity"].notna()].copy()
    
    GK = ["Reporter Country Code", "Reporter Countries", "Commodity", "Year"]
    FK = GK + ["Partner Country Code", "Partner Countries"]
    
    flows = imp.groupby(FK, as_index=False)["Value"].sum().rename(
        columns={"Value": "import_t"}
    )
    tot = flows.groupby(GK, as_index=False)["import_t"].sum().rename(
        columns={"import_t": "total_t"}
    )
    flows = flows.merge(tot, on=GK, how="left")
    flows["share"] = flows["import_t"] / flows["total_t"]
    
    def shannon_h(s):
        p = s[s > 0].values
        return -np.sum(p * np.log(p)) if len(p) > 1 else 0.0
    
    conc = flows.groupby(GK, as_index=False).agg(
        partner_hhi=("share", lambda x: (x ** 2).sum()),
        shannon_h=("share", shannon_h),
        partner_count=("Partner Country Code", "nunique"),
        top_partner_share=("share", "max"),
        suppliers_over_5pct=("share", lambda x: (x > 0.05).sum()),
        total_import_quantity_t=("import_t", "sum"),
    )
    conc["effective_suppliers_hhi"] = 1.0 / conc["partner_hhi"]
    conc["effective_suppliers_shannon"] = np.exp(conc["shannon_h"])
    conc["shannon_evenness"] = np.where(
        conc["partner_count"] > 1,
        conc["shannon_h"] / np.log(conc["partner_count"]),
        np.nan,
    )

print(f"Concentration data: {len(conc):,} rows")
print(f"Columns: {conc.columns.tolist()}")

## 2. Full Correlation Matrix (All Observations)

This is the primary output — the matrix that defends the index design choice.

In [ ]:
# ---- Select the metrics to correlate ----
# These are all the concentration/diversity metrics available.
metric_cols = [
    "partner_hhi",
    "shannon_h",
    "shannon_evenness",
    "top_partner_share",
    "partner_count",
    "suppliers_over_5pct",
    "effective_suppliers_hhi",
    "effective_suppliers_shannon",
]

# Use only columns that exist (in case of partial data)
available_cols = [c for c in metric_cols if c in conc.columns]

# ---- Compute Pearson correlation ----
corr_all = conc[available_cols].corr()

print("FULL CORRELATION MATRIX (all commodity-country-years)")
print("=" * 60)
print(corr_all.round(3).to_string())

In [ ]:
# ---- Heatmap ----
# Use human-readable labels
label_map = {
    "partner_hhi": "HHI",
    "shannon_h": "Shannon H",
    "shannon_evenness": "Shannon\nEvenness",
    "top_partner_share": "Top-1\nShare",
    "partner_count": "Partner\nCount",
    "suppliers_over_5pct": "Suppliers\n>5%",
    "effective_suppliers_hhi": "Eff. Suppliers\n(1/HHI)",
    "effective_suppliers_shannon": "Eff. Suppliers\n(e^H)",
}

labels = [label_map.get(c, c) for c in available_cols]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_all, dtype=bool), k=1)  # upper triangle mask

sns.heatmap(
    corr_all,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    xticklabels=labels,
    yticklabels=labels,
    ax=ax,
    cbar_kws={"label": "Pearson r", "shrink": 0.8},
)

ax.set_title("Concentration Metric Correlations\n(All commodity-country-years)", fontsize=14)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "concentration_correlation_heatmap.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 3. Per-Commodity Correlations

Check whether the correlation structure varies across Wheat, Rice, and Maize.  
If it does, it means the metrics behave differently for different market  
structures.

In [ ]:
# ---- Key correlation pairs to track across commodities ----
key_pairs = [
    ("partner_hhi", "top_partner_share", "HHI vs Top-1 Share"),
    ("partner_hhi", "shannon_h", "HHI vs Shannon H"),
    ("shannon_h", "top_partner_share", "Shannon H vs Top-1 Share"),
    ("effective_suppliers_hhi", "effective_suppliers_shannon", "Eff. Suppliers: HHI vs Shannon"),
    ("partner_hhi", "partner_count", "HHI vs Partner Count"),
    ("shannon_h", "partner_count", "Shannon H vs Partner Count"),
]

pair_results = []

for col_a, col_b, label in key_pairs:
    if col_a not in conc.columns or col_b not in conc.columns:
        continue
    
    # Overall correlation
    r_all = conc[[col_a, col_b]].corr().iloc[0, 1]
    
    row = {"Metric Pair": label, "All": r_all}
    
    # Per-commodity correlation
    for commodity in COMMODITIES:
        sub = conc[conc["Commodity"] == commodity]
        r = sub[[col_a, col_b]].corr().iloc[0, 1]
        row[commodity] = r
    
    pair_results.append(row)

pair_df = pd.DataFrame(pair_results)
print("KEY CORRELATION PAIRS BY COMMODITY")
print("=" * 70)
print(pair_df.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
# ---- Per-commodity heatmaps side by side ----
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, commodity in zip(axes, COMMODITIES):
    sub = conc[conc["Commodity"] == commodity][available_cols]
    corr_c = sub.corr()
    mask = np.triu(np.ones_like(corr_c, dtype=bool), k=1)
    
    sns.heatmap(
        corr_c,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
        cbar=False,
    )
    ax.set_title(commodity, fontsize=13, weight="bold")

fig.suptitle("Concentration Metric Correlations by Commodity", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "correlation_heatmap_by_commodity.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 4. The Key Question: Does Shannon Add Information Beyond HHI?

Shannon is worth adding to the analysis toolkit only if it captures something  
HHI does not.
1.  Identifying countries where HHI and Shannon **disagree** most — these are  
   the cases where Shannon reveals structure that HHI misses.

In [ ]:
# ---- 4a. Correlation comparison ----
print("DOES SHANNON ADD INFORMATION BEYOND HHI?")
print("=" * 55)

if "top_partner_share" in conc.columns and "shannon_h" in conc.columns:
    r_hhi_top1 = conc[["partner_hhi", "top_partner_share"]].corr().iloc[0, 1]
    r_hhi_shannon = conc[["partner_hhi", "shannon_h"]].corr().iloc[0, 1]
    
    print(f"\n  |r(HHI, Top-1 Share)|    = {abs(r_hhi_top1):.4f}")
    print(f"  |r(HHI, Shannon H)|      = {abs(r_hhi_shannon):.4f}")
    print(f"  Difference               = {abs(r_hhi_top1) - abs(r_hhi_shannon):.4f}")
    
    if abs(r_hhi_shannon) < abs(r_hhi_top1):
        print("\n  --> Shannon has LOWER correlation with HHI than Top-1 does.")
        print("      Shannon captures MORE independent information.")
        print("      This supports including Shannon as a complementary metric.")
    else:
        print("\n  --> Shannon is MORE correlated with HHI than Top-1.")
        print("      Shannon adds LESS independent information than Top-1.")
        print("      Shannon is still useful for interpretation but should not")
        print("      be added to the composite index alongside HHI.")

In [ ]:
# ---- 4b. Where HHI and Shannon disagree most ----
# Rank by HHI and by Shannon, then find large rank differences.
# These are the countries where the choice of metric changes the story.

for commodity in COMMODITIES:
    sub = conc[conc["Commodity"] == commodity].copy()
    if sub.empty:
        continue
    
    # Use 2021-2023 average
    sub_w = sub[sub["Year"].between(2021, 2023)].copy()
    avg = sub_w.groupby(
        ["Reporter Country Code", "Reporter Countries"], as_index=False
    ).agg(
        mean_hhi=("partner_hhi", "mean"),
        mean_shannon=("shannon_h", "mean"),
        mean_count=("partner_count", "mean"),
    )
    
    # Rank: lower rank = more concentrated (worse) for HHI,
    #        lower rank = less diverse (worse) for Shannon
    avg["rank_hhi"] = avg["mean_hhi"].rank(ascending=False)  # high HHI = high rank
    avg["rank_shannon"] = avg["mean_shannon"].rank(ascending=True)  # low Shannon = high rank
    avg["rank_diff"] = abs(avg["rank_hhi"] - avg["rank_shannon"])
    
    divergent = avg.nlargest(10, "rank_diff")
    
    print(f"\n{'='*60}")
    print(f"{commodity}: Countries where HHI and Shannon rank DISAGREE most")
    print(f"{'='*60}")
    print("(Large rank difference = metric choice changes the story)")
    print(
        divergent[
            ["Reporter Countries", "mean_hhi", "mean_shannon", "mean_count",
             "rank_hhi", "rank_shannon", "rank_diff"]
        ].to_string(index=False, float_format="{:.3f}".format)
    )

## 5. Scatter Matrix — Pairwise Relationships

A visual overview of how all the concentration metrics relate to each other.

In [ ]:
# ---- Pairwise scatter for the 4 main metrics ----
scatter_cols = ["partner_hhi", "shannon_h", "top_partner_share", "partner_count"]
scatter_labels = ["HHI", "Shannon H", "Top-1 Share", "Partner Count"]

scatter_available = [c for c in scatter_cols if c in conc.columns]
scatter_labels_available = [
    scatter_labels[scatter_cols.index(c)] for c in scatter_available
]

# Sample for performance (scatter matrix with 50k+ points is unreadable)
sample = conc[scatter_available + ["Commodity"]].dropna().sample(
    min(5000, len(conc)), random_state=42
)

g = sns.pairplot(
    sample,
    vars=scatter_available,
    hue="Commodity",
    plot_kws={"alpha": 0.3, "s": 10, "edgecolor": "none"},
    diag_kws={"alpha": 0.5},
    height=2.5,
)

# Relabel axes
for i, label in enumerate(scatter_labels_available):
    g.axes[-1, i].set_xlabel(label, fontsize=9)
    g.axes[i, 0].set_ylabel(label, fontsize=9)

g.figure.suptitle("Pairwise Scatter: Concentration Metrics", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "concentration_pairplot.png"), dpi=120, bbox_inches="tight")
plt.show()

## 6. Summary & Recommendations

In [ ]:
print("=" * 60)
print("SUMMARY & DESIGN RECOMMENDATIONS")
print("=" * 60)

if "top_partner_share" in conc.columns and "shannon_h" in conc.columns:
    r_hhi_top1 = abs(conc[["partner_hhi", "top_partner_share"]].corr().iloc[0, 1])
    r_hhi_shan = abs(conc[["partner_hhi", "shannon_h"]].corr().iloc[0, 1])
    r_top1_shan = abs(conc[["top_partner_share", "shannon_h"]].corr().iloc[0, 1])
    
    print(f"\n  |r(HHI, Top-1)|     = {r_hhi_top1:.3f}")
    print(f"  |r(HHI, Shannon)|   = {r_hhi_shan:.3f}")
    print(f"  |r(Top-1, Shannon)| = {r_top1_shan:.3f}")
    
    print("\n  DESIGN DECISIONS (empirically supported):")
    print(f"  1. HHI and Top-1 at r={r_hhi_top1:.2f}: DO NOT include both")
    print(f"     in the composite index (double-weights concentration).")
    print(f"  2. Use HHI in the index (full distribution information).")
    print(f"     Report Top-1 in narrative/charts (intuitive for non-experts).")
    
    if r_hhi_shan < r_hhi_top1 - 0.05:
        print(f"  3. Shannon H at r={r_hhi_shan:.2f} with HHI adds independent")
        print(f"     information. It captures mid-distribution structure that")
        print(f"     HHI misses. Use it as a complementary diagnostic.")
    else:
        print(f"  3. Shannon H at r={r_hhi_shan:.2f} with HHI is similarly")
        print(f"     correlated. It remains useful for interpretation but")
        print(f"     should not be added to the composite alongside HHI.")
    
    print(f"  4. Partner count is the most independent metric (lowest r with")
    r_hhi_count = abs(conc[["partner_hhi", "partner_count"]].corr().iloc[0, 1])
    print(f"     HHI: r={r_hhi_count:.2f}). Its inclusion in the index at")
    print(f"     weight 1 (vs HHI weight 2) is justified.")

print("\nVisualizations saved:")
print("  - concentration_correlation_heatmap.png")
print("  - correlation_heatmap_by_commodity.png")
print("  - concentration_pairplot.png")